In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score
from imblearn.over_sampling import SMOTE


df = pd.read_csv("D:/Đại học/1. Nghiên cứu Khoa học/Dataset/Folder_Dataset/P2P_Dataset.csv")
boolean_cols = df.select_dtypes(include=['bool']).columns
df[boolean_cols] = df[boolean_cols].astype(int)
df = df.drop(columns=["badloan", "funded_amnt", "sub_grade", "pub_rec", "num_tl_30dpd"])


In [ ]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score

In [ ]:
# 🎯 **Chia X, y**
X = df.drop(columns=["default_binary"], errors="ignore")
y = df["default_binary"]

# 🔀 **Chia tập train/test**
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=50, stratify=y)

# ⚖️ **Cân bằng dữ liệu bằng SMOTE**
smote = SMOTE(sampling_strategy=0.5, random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

In [ ]:
# 📌 **Danh sách các biến quan trọng từ Feature Importance**
selected_features_rf = [
    "acc_open_past_24mths", "annual_inc", "delinq_2yrs", "dti", "earnings",
    "emp_length", "grade", "inq_last_12m", "inq_last_6mths", "installment",
    "int_rate", "loan_amnt", "loan_vol6m", "mort_acc", "mths_since_last_delinq",
    "num_accts_ever_120_pd", "num_actv_rev_tl", "open_acc", "pct_tl_nvr_dlq",
    "pub_rec_bankruptcies", "revol_util", "term", "home_ownership_MORTGAGE",
    "home_ownership_OWN", "home_ownership_RENT", "purpose_credit_card",
    "purpose_debt_consolidation", "purpose_home_improvement", "purpose_other"
]

# ✅ **Lọc dữ liệu đúng cách (Dùng tập đã qua SMOTE)**
X_train_selected_rf = X_train_resampled[selected_features_rf]  # Dữ liệu đã được SMOTE
X_test_selected_rf = X_test[selected_features_rf]  # Giữ nguyên test set

# 🔄 **Chuẩn hóa lại dữ liệu**
scaler = StandardScaler()
X_train_selected_rf_scaled = scaler.fit_transform(X_train_selected_rf)
X_test_selected_rf_scaled = scaler.transform(X_test_selected_rf)



# 🔍 Lấy độ quan trọng của từng biến
importance_rf = rf_model.feature_importances_

# 📊 Hiển thị tầm quan trọng của biến
for i, v in enumerate(importance_rf):
    print(f'Biến: {X_train.columns[i]}, Tầm quan trọng: {v:.5f}')

# 🎨 Vẽ biểu đồ Feature Importance
plt.figure(figsize=(12, 6))
plt.bar([x for x in range(len(importance_rf))], importance_rf)
plt.xticks(range(len(importance_rf)), X_train.columns, rotation=90)
plt.xlabel("Features")
plt.ylabel("Importance Score")
plt.title("Feature Importance từ Random Forest")
plt.show()


# 🎯 Chỉ chọn các biến có tầm quan trọng > 0.01
important_features_rf = X_train.columns[importance_rf > 0.01]

# 🔄 Cập nhật X_train và X_test chỉ với các biến quan trọng
X_train_selected = X_train_resampled[important_features_rf]
X_test_selected = X_test[important_features_rf]


# 🔄 Chuẩn hóa dữ liệu
scaler = StandardScaler()
X_train_selected_scaled = scaler.fit_transform(X_train_selected)
X_test_selected_scaled = scaler.transform(X_test_selected)


In [ ]:
# 🛠 **Giảm tải bộ nhớ - Giới hạn tập siêu tham số**
param_dist = {
    "n_estimators": [50, 100],  # Giảm số cây quyết định để giảm RAM
    "max_depth": [5, 10, None],  # Giảm số độ sâu
    "min_samples_split": [5, 10],  # Số mẫu tối thiểu để tách
    "min_samples_leaf": [1, 3],  # Số mẫu tối thiểu tại mỗi lá
    "class_weight": ["balanced"],  # Cân bằng dữ liệu
    "bootstrap": [True]  # Chỉ dùng bootstrap để giảm bộ nhớ
}

# 🚀 **Randomized Search**
random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=5,  # Giảm xuống 5 để giảm thời gian
    scoring="roc_auc",
    cv=2,  # Giảm số lần cross-validation để nhanh hơn
    n_jobs=-1,  # Chạy song song
    random_state=42
)

# 🔥 **Huấn luyện mô hình và tìm tham số tối ưu**
random_search.fit(X_train_selected_rf_scaled, y_train_resampled)

# 📌 **Lấy siêu tham số tốt nhất**
best_params = random_search.best_params_
print("\n🔥 Best Hyperparameters for Random Forest:")
print(best_params)


🔥 Best Hyperparameters for Random Forest:
{'n_estimators': 50, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_depth': None, 'class_weight': 'balanced', 'bootstrap': True}


In [ ]:
# 🏆 **Huấn luyện lại mô hình với siêu tham số tối ưu**
best_rf_model = RandomForestClassifier(**best_params, random_state=42)
best_rf_model.fit(X_train_selected_rf_scaled, y_train_resampled)

# 📊 **Dự đoán trên tập kiểm tra**
y_pred_rf = best_rf_model.predict(X_test_selected_rf_scaled)
y_prob_rf = best_rf_model.predict_proba(X_test_selected_rf_scaled)[:, 1]

# 🎯 **Đánh giá mô hình**
accuracy_rf = accuracy_score(y_test, y_pred_rf)
roc_auc_rf = roc_auc_score(y_test, y_prob_rf)
conf_matrix_rf = confusion_matrix(y_test, y_pred_rf)
report_rf = classification_report(y_test, y_pred_rf)

# 📊 **In kết quả**
print("\n📊 Random Forest Model (Feature Selected)")
print(f"🎯 Accuracy: {accuracy_rf:.4f}")
print(f"🚀 AUC-ROC: {roc_auc_rf:.4f}")
print(f"📑 Classification Report:\n{report_rf}")
print(f"📊 Confusion Matrix:\n{conf_matrix_rf}")


📊 Random Forest Model (Feature Selected)
🎯 Accuracy: 0.9061
🚀 AUC-ROC: 0.7293
📑 Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.99      0.95    494039
           1       0.28      0.05      0.09     46647

    accuracy                           0.91    540686
   macro avg       0.60      0.52      0.52    540686
weighted avg       0.86      0.91      0.88    540686

📊 Confusion Matrix:
[[487379   6660]
 [ 44098   2549]]
